In [ ]:
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import matplotlib.pyplot as plt
import Plot_setting
import numpy as np
from scipy.special import jv
from scipy.integrate import solve_ivp, solve_bvp
from scipy.interpolate import interp1d

# ---------------- physical parameters ----------------
phi_a = 0.11
phi_b = 0.204
E_J = 37.12 * 2 * np.pi
kappa_b = 3 / 10.4
epsilon_p = 0.1
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
eps_d = 2 * g2
g = E_J * np.sin(epsilon_p) * phi_a**2 * phi_b

A_0 = np.sqrt(eps_d / g)
c = np.sqrt(2) * g
kappa_a = 0.0   # <-- this file: the exactly-solvable case

# exact fixed points at kappa_a = 0 (order: A, PA, B, PB)
W_p = np.array([A_0, 0.0, 0.0, 0.0])
M_p = np.array([0.0, A_0, 0.0, 0.0])

# ---------------- dynamics ----------------
def F_loss_a(X):
    A, PA, B, PB = X
    dA  = -c*(A*B - PA*PB) - kappa_a*(A/2 + PA)
    dPA =  -c*(A*PB - PA*B) + kappa_a*PA/2
    dB  =  -eps_d/np.sqrt(2) + g*(A**2 + PA**2)/np.sqrt(2) - kappa_b*(B/2 + PB)
    dPB =   c*A*PA + kappa_b*PB/2
    return np.array([dA, dPA, dB, dPB])

def jacobian(X):
    A, PA, B, PB = X
    return np.array([
        [-c*B - kappa_a/2, c*PB - kappa_a, -c*A, c*PA],
        [-c*PB, c*B + kappa_a/2, c*PA, -c*A],
        [c*A, c*PA, -kappa_b/2, -kappa_b],
        [c*PA, c*A, 0, kappa_b/2]
    ])

# ---------------- boundary conditions ----------------
# leave W_p along its unstable manifold, arrive at M_p along its stable manifold
# -> project the deviation onto each point's own eigenbasis and kill the "wrong" components
JW = jacobian(W_p); eigvals_W, eigvecs_W = np.linalg.eig(JW)
JM = jacobian(M_p); eigvals_M, eigvecs_M = np.linalg.eig(JM)
stable_idx_W   = np.where(eigvals_W.real < 0)[0]   # 2 conditions
unstable_idx_M = np.where(eigvals_M.real > 0)[0]   # 2 conditions
Vinv_W = np.linalg.inv(eigvecs_W)
Vinv_M = np.linalg.inv(eigvecs_M)

def bc(ya, yb):
    ca = (Vinv_W @ (ya - W_p))[stable_idx_W].real
    cb = (Vinv_M @ (yb - M_p))[unstable_idx_M].real
    return np.concatenate([ca, cb])            # shape (4,) -- matches n=4, no free params

def fun(t, Y):
    return F_loss_a(Y)

# ---------------- build the initial guess exactly, via Gamma ----------------
# On Gamma: P_A = A_0 - A, P_B = -B, and (A,B) obeys a clean 2D reduced system.
# M_p (A=0,B=0 in these reduced coords) has ONE well-defined stable eigendirection
# within this 2D system -- shoot backward along it (only 2 signs to try, no angle search).
D_M = np.sqrt(kappa_b**2 + 16*c**2*A_0**2)
lam_s = (kappa_b - D_M) / 4
v1, v2 = 1.0, -lam_s/(c*A_0)
norm = np.hypot(v1, v2)
v1, v2 = v1/norm, v2/norm

def rhs2d_backward(t, y):
    A, B = y
    dA = -c*A_0*B
    dB = c*A*(A - A_0) + kappa_b/2*B
    return [-dA, -dB]     # backward time

sol2d = solve_ivp(rhs2d_backward, [0, 900], [1e-6*v1, 1e-6*v2],
                   rtol=1e-12, atol=1e-15, max_step=2.0)
# (the sign above converges to W_p; the other sign diverges -- check both if this ever needs re-deriving)

# reverse time order (we shot backward from M_p) so it runs W_p -> M_p
t_rev = sol2d.t[-1] - sol2d.t[::-1]
A_rev, B_rev = sol2d.y[0, ::-1], sol2d.y[1, ::-1]
PA_rev, PB_rev = A_0 - A_rev, -B_rev
orbit4d = np.stack([A_rev, PA_rev, B_rev, PB_rev], axis=1)

# resample uniformly in arc length; trim the (very long) flat tails near each fixed point
speed = np.linalg.norm(np.diff(orbit4d, axis=0), axis=1)
cum = np.concatenate([[0], np.cumsum(speed)])
lo, hi = 0.0, cum[-1]          # trim=0 worked cleanly here; try e.g. 0.02*cum[-1] if you shorten the domain
N = 100
s_uniform = np.linspace(lo, hi, N)
interp = lambda col: interp1d(cum, orbit4d[:, col])(s_uniform)
y_guess = np.stack([interp(0), interp(1), interp(2), interp(3)])
x_guess = interp1d(cum, t_rev)(s_uniform)
x_guess -= x_guess[0]

# ---------------- solve ----------------
res = solve_bvp(fun, bc, x_guess, y_guess, tol=1e-6, max_nodes=20000)
print(res.status, res.message, "max residual:", np.max(res.rms_residuals))
print("y(0) =", res.y[:, 0], " (target W_p =", W_p, ")")
print("y(-1)=", res.y[:, -1], " (target M_p =", M_p, ")")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(res.y[0], res.y[2]); axes[0].set_xlabel("A"); axes[0].set_ylabel("B")
axes[1].plot(res.y[1], res.y[3]); axes[1].set_xlabel("P_A"); axes[1].set_ylabel("P_B")
plt.tight_layout(); plt.show()

In [ ]:
# ---------------- physical parameters ----------------
c=np.sqrt(2)*g
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2

delta = kappa_a * kappa_b / 4 / g**2
A_0_p = np.sqrt(A_0**2-delta)
a = (A_0**2)/(2*A_0**2 - 2*delta)*(1 + (2*delta)/(A_0**2)-np.sqrt(1+4*(2*delta)/(A_0**2)))
P_A_value = np.sqrt(delta/(a*(a**2-1)))
A_value = a*P_A_value
B_value = -2*np.sqrt(2)*g/kappa_b*A_value**2-kappa_a/(2*np.sqrt(2)*g)
P_B_value = -2*np.sqrt(2)*g*A_value*P_A_value/kappa_b

W_p = [A_0_p, 0, -kappa_a/(2*np.sqrt(2)*g), 0]
M_p = [A_value, P_A_value, B_value, P_B_value]
S = [0, 0, -np.sqrt(2)*eps_d/kappa_b, 0]


# ---------------- dynamics ----------------


def F_loss_a(X):
    A, PA, B, PB = X
    dA = - c*(A*B - PA*PB) - kappa_a * (A/2+PA)
    dPA =   - c * (A * PB - PA * B) + kappa_a * PA/2
    dB =  - eps_d/np.sqrt(2) + g*(A**2 + PA**2)/np.sqrt(2) - kappa_b * (B/2 + PB)
    dPB =  c * A * PA + kappa_b * PB/2
    return np.array([dA, dPA, dB, dPB])

def jacobian(X):
    A, PA, B, PB = X

    J = np.array([
        [-c*B-kappa_a/2, c*PB-kappa_a, -c*A, c*PA],
        [-c*PB, c*B+kappa_a/2, c*PA, -c*A],
        [c*A, c*PA, -kappa_b/2, -kappa_b],
        [c*PA, c*A, 0, kappa_b/2]             
    ])

    return J

J_W_p = jacobian(W_p)
eigvals_W_p, eigvecs_W_p = np.linalg.eig(J_W_p)
J_M_p = jacobian(M_p)
eigvals_M_p, eigvecs_M_p = np.linalg.eig(J_M_p)
J_S = jacobian(S)
eigvals_S, eigvecs_S = np.linalg.eig(J_S)

eps_fp = 1e-3

A_init = A_0 - eps_fp
PA_init = eps_fp
B_init = np.sqrt(2)*g/kappa_b * (A_init**2+4*A_init*PA_init+PA_init**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g
PB_init = -2*np.sqrt(2)*g/kappa_b*A_init*PA_init
BC_init = [A_init, PA_init, B_init, PB_init]

A_fin = A_value - eps_fp
PA_fin = P_A_value - eps_fp
B_fin = np.sqrt(2)*g/kappa_b * (A_fin**2+4*A_fin*PA_fin+PA_fin**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g
PB_fin = -2*np.sqrt(2)*g/kappa_b*A_fin*PA_fin
BC_fin = [A_fin, PA_fin, B_fin, PB_fin]

bc = [BC_init, BC_fin]

res = solve_bvp(F_loss_a, bc, x, y_guess)

In [ ]:
"""
3D visualization of the driven-dissipative two-mode flow, restricted to the
energy shell H = 0, for kappa_a != 0 -- THIS VERSION uses (A, P_A, P_B) as
the plotted axes and solves H=0 for B (instead of the other way around).

Unlike P_B (which enters H quadratically, giving two branches), B enters H
*linearly*:
    H = B*(-sqrt2 g A P_A - kappa_b/2 P_B) + (terms without B)
so H=0 has a UNIQUE solution for B given (A, P_A, P_B) -- no branch
ambiguity -- except where the coefficient of B vanishes (measure zero, and
avoided below by simply not counting a point as a valid seed if the solve
fails or blows up).

Because this coefficient can be small away from that measure-zero set, the
solved B can occasionally be numerically huge for "generic" random
(A, P_A, P_B) that don't correspond to physically relevant states. We
discard any solution with |B_sol| > 10*|B_0| as unphysical/irrelevant for
this visualization (kept configurable via B_CUTOFF below).

Phase space is 4D: (A, B, P_A, P_B). H is exactly conserved (verified below
by finite differences), so any trajectory that starts on {H=0} stays there
forever.

IMPORTANT, checked numerically (and true for essentially any kappa_a != 0,
not just the specific value below -- H(M+') scales approximately linearly
with kappa_a for small kappa_a, and is ~-0.0159 at the parameters given,
where delta/A_0^2 ~ 0.5, about half way to the well-merging threshold): the
shifted wells W+', W-' and the saddle S sit exactly on H=0, but M+', M-' do
NOT. Since H is *exactly* conserved along every trajectory, this by itself
already proves no orbit can connect W+' to M+' for any kappa_a != 0 -- a
clean energy-conservation argument, independent of (and simpler than) the
Groebner-basis proof that Gamma/Sigma stop being exact invariant surfaces.
The plot marks M+'/M-' as off-shell reference points rather than
pretending a heteroclinic connection to them exists on this shell.

The plot shows:
  - W+', W-', S (exactly on H=0) as filled markers, with pieces of their
    unstable manifolds (forward in time, red) and stable manifolds
    (backward in time, green)
  - M+', M-' as open/faded markers with an annotation that they are off
    this shell
  - a scatter of short forward/backward flow lines seeded at random points
    on the H=0 shell (B solved from H=0, points with |B|>10|B_0| discarded),
    to show the general flow direction near the shell
"""

import numpy as np
from scipy.special import jv
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

rng = np.random.default_rng(0)

# ---------------- physical parameters ----------------
epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 2
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g

A_0 = np.sqrt(eps_d / g)
c = np.sqrt(2) * g

delta = kappa_a * kappa_b / 4 / g**2
A_0_p = np.sqrt(A_0**2 - delta)
a = (A_0**2) / (2 * A_0**2 - 2 * delta) * (
    1 + (2 * delta) / (A_0**2) - np.sqrt(1 + 4 * (2 * delta) / (A_0**2))
)
P_A_value = np.sqrt(delta / (a * (a**2 - 1)))
A_value = a * P_A_value
B_value = -2 * np.sqrt(2) * g / kappa_b * A_value**2 - kappa_a / (2 * np.sqrt(2) * g)
P_B_value = -2 * np.sqrt(2) * g * A_value * P_A_value / kappa_b

# ---------------- fixed points (A,B,P_A,P_B) ----------------
Mp = np.array([A_value, B_value, P_A_value, P_B_value])
Mm = np.array([-A_value, B_value, -P_A_value, P_B_value])          # mirror symmetry
Wp = np.array([A_0_p, -kappa_a / (2 * np.sqrt(2) * g), 0.0, 0.0])
Wm = np.array([-A_0_p, -kappa_a / (2 * np.sqrt(2) * g), 0.0, 0.0])
B0 = -np.sqrt(2) * eps_d / kappa_b
S = np.array([0.0, B0, 0.0, 0.0])

fixed_points = {"W+'": Wp, "W-'": Wm, "S": S, "M+'": Mp, "M-'": Mm}

# ---------------- equations of motion (kappa_a != 0) ----------------
def rhs(t, y):
    A, B, PA, PB = y
    dA = np.sqrt(2) * g * (PA * PB - A * B) - kappa_a / 2 * A - kappa_a * PA
    dB = g / np.sqrt(2) * (A**2 + PA**2) - eps_d / np.sqrt(2) - kappa_b / 2 * B - kappa_b * PB
    dPA = np.sqrt(2) * g * (B * PA - A * PB) + kappa_a / 2 * PA
    dPB = np.sqrt(2) * g * A * PA + kappa_b / 2 * PB
    return [dA, dB, dPA, dPB]

def H(y):
    A, B, PA, PB = y
    return (
        -np.sqrt(2) * g * A * B * PA
        + (g * A**2 / np.sqrt(2) - eps_d / np.sqrt(2) - kappa_b * B / 2) * PB
        - kappa_b / 2 * PB**2
        + g / np.sqrt(2) * PA**2 * PB
        - kappa_a / 2 * PA**2
        - kappa_a / 2 * A * PA
    )

# sanity check: fixed points should have H=0
for name, fp in fixed_points.items():
    print(f"H({name}) = {H(fp):.3e}")

# ---------------- solve H=0 for B given (A,P_A,P_B) [H is LINEAR in B] ----------------
B_CUTOFF = 10 * abs(B0)

def solve_B(A, PA, PB):
    coefB = -np.sqrt(2) * g * A * PA - kappa_b / 2 * PB
    const = (
        (g * A**2 / np.sqrt(2) - eps_d / np.sqrt(2)) * PB
        - kappa_b / 2 * PB**2
        + g / np.sqrt(2) * PA**2 * PB
        - kappa_a / 2 * PA**2
        - kappa_a / 2 * A * PA
    )
    if abs(coefB) < 1e-13:
        return None  # degenerate: coefficient of B vanishes, no unique solution
    B_sol = -const / coefB
    if abs(B_sol) > B_CUTOFF:
        return None  # discard: unphysically large, not a relevant seed point
    return B_sol

# ---------------- finite-difference Jacobian + eigen-directions ----------------
def jacobian(y, eps=1e-7):
    n = len(y)
    J = np.zeros((n, n))
    f0 = np.array(rhs(0, y))
    for i in range(n):
        yp = np.array(y, dtype=float)
        yp[i] += eps
        J[:, i] = (np.array(rhs(0, yp)) - f0) / eps
    return J

def unstable_stable_dirs(y):
    J = jacobian(y)
    evals, evecs = np.linalg.eig(J)
    unstable, stable = [], []
    for k in range(len(evals)):
        lam = evals[k]
        v = evecs[:, k]
        if abs(lam.imag) > 1e-8:
            continue  # skip complex pairs for simple seeding
        v = v.real / np.linalg.norm(v.real)
        if lam.real > 1e-8:
            unstable.append(v)
        elif lam.real < -1e-8:
            stable.append(v)
    return unstable, stable

# ---------------- integration helper (stops on blow-up) ----------------
BOX = 3 * max(A_0_p, abs(B0), P_A_value) + 1  # generous bounding box

def blowup(t, y):
    return np.linalg.norm(y) - 10 * BOX
blowup.terminal = True
blowup.direction = 1

def integrate(y0, tmax, sign=+1):
    sol = solve_ivp(
        lambda t, y: [sign * v for v in rhs(t, y)],
        [0, tmax], y0, max_step=0.05, rtol=1e-9, atol=1e-11,
        events=blowup, dense_output=False,
    )
    return sol.y  # shape (4, npoints)

# ---------------- build trajectories near each fixed point ----------------
traj_unstable = []  # forward in time, along unstable eigvecs
traj_stable = []    # backward in time, along stable eigvecs

EPS_SEED = 1e-4
TMAX = 60.0

on_shell_names = [name for name, fp in fixed_points.items() if abs(H(fp)) < 1e-6]
off_shell_names = [name for name in fixed_points if name not in on_shell_names]
print("On the H=0 shell:", on_shell_names)
print("Off the H=0 shell (different energy):", off_shell_names,
      {n: H(fixed_points[n]) for n in off_shell_names})

for name in on_shell_names:
    fp = fixed_points[name]
    u_dirs, s_dirs = unstable_stable_dirs(fp)
    for v in u_dirs:
        for sgn in (+1, -1):
            y0 = fp + sgn * EPS_SEED * v
            Y = integrate(y0, TMAX, sign=+1)
            traj_unstable.append((name, Y))
    for v in s_dirs:
        for sgn in (+1, -1):
            y0 = fp + sgn * EPS_SEED * v
            Y = integrate(y0, TMAX, sign=-1)  # backward in time
            traj_stable.append((name, Y))



In [ ]:
A_all   = np.arange(-5.2, 5.2, 0.01)
PA_all  = np.arange(-5.2, 5.2, 0.01)
A_mesh, PA_mesh   = np.meshgrid(A_all, PA_all)
H_vals = (-PA_mesh**2 * kappa_a / 2
          + ((A_0**2 - delta) * A_mesh * PA_mesh - A_mesh**3 * PA_mesh - A_mesh * PA_mesh**3 - 2 * A_mesh**2 * PA_mesh**2)
          * 2 * g**2 / kappa_b)
cs = plt.contour(A_all, PA_all, H_vals, levels=[0.0], colors='white', linewidths=2)
paths = cs.get_paths()
contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)
begin = 3780
end = 4173
A_contour  = contour_coords[:, 0][begin:end]
PA_contour = contour_coords[:, 1][begin:end]
PB_contour = -2*np.sqrt(2)*g/kappa_b*A_contour*PA_contour
B_contour = np.sqrt(2)*g/kappa_b * (A_contour**2+4*A_contour*PA_contour+PA_contour**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g
fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.scatter(A_contour,PA_contour, c=range(len(A_contour)))
ax_1.set_xlabel(r'$A$')
ax_1.set_ylabel(r'$P_A$')

sc = ax_2.scatter(B_contour, PB_contour, c=range(len(B_contour)))
ax_2.set_xlabel(r'$B$')
ax_2.set_ylabel(r'$P_B$')

plt.colorbar(sc, ax=ax_2)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------- general flow field: random seeds on H=0 ----------------
flow_lines = []
N_SEED = 1_000
n_tries = 0
while len(flow_lines) < N_SEED and n_tries < 20 * N_SEED:
    n_tries += 1
    Aa = rng.uniform(-1.3 * A_0_p, 1.3 * A_0_p)
    PAa = rng.uniform(-1.3 * P_A_value, 1.3 * P_A_value)
    PBa = rng.uniform(-1.3 * abs(P_B_value), 1.3 * abs(P_B_value))
    Bb = solve_B(Aa, PAa, PBa)
    if Bb is None:
        continue
    y0 = np.array([Aa, Bb, PAa, PBa])
    Yf = integrate(y0, 6.0, sign=+1)
    #Yb = integrate(y0, 6.0, sign=-1)
    flow_lines.append(Yf)
    print(len(flow_lines))
print(f"Kept {len(flow_lines)}/{n_tries} random seeds after the |B|>{B_CUTOFF:.3f} cutoff")


In [ ]:

%matplotlib widget
# ---------------- plot ----------------
fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection="3d")

# thin general flow lines (light gray/blue), showing local flow direction
#for Yb, Yf in flow_lines:
#    ax.plot(Yb[0], Yb[1], Yb[2], color="0.75", lw=0.6, alpha=0.7)
#    ax.plot(Yf[0], Yf[1], Yf[2], color="steelblue", lw=0.6, alpha=0.7)

ax.plot(A_contour, B_contour, PA_contour, color="black", lw=2)

for Yf in flow_lines:
    ax.plot(Yf[0], Yf[1], Yf[2], color="steelblue", lw=0.6, alpha=0.7)
    #ax.scatter(*Yf[:3, 0], color = "red", s=.5)
# unstable-manifold pieces (forward), stable-manifold pieces (backward)
for name, Y in traj_unstable:
    ax.plot(Y[0], Y[1], Y[2], color="crimson", lw=1.6)
for name, Y in traj_stable:
    ax.plot(Y[0], Y[1], Y[2], color="darkgreen", lw=1.6)

# fixed points: solid = exactly on H=0, hollow/faded = off this shell
colors = {"W+'": "black", "W-'": "black", "S": "purple", "M+'": "orange", "M-'": "orange"}
for name, fp in fixed_points.items():
    if name in on_shell_names:
        ax.scatter(*fp[:3], color=colors[name], s=80, depthshade=False, zorder=5)
        ax.text(fp[0], fp[1], fp[2], f"  {name}", fontsize=10, weight="bold")
    else:
        ax.scatter(*fp[:3], facecolors="none", edgecolors=colors[name], s=90,
                   linewidths=1.8, depthshade=False, zorder=5)
        ax.text(fp[0], fp[1], fp[2], f"  {name} (off shell, H={H(fp):.3f})",
                fontsize=8, style="italic", color=colors[name])

ax.set_xlabel("A")
ax.set_ylabel("B")
ax.set_zlabel("P_A")
ax.set_xlim(-2*A_0_p,2*A_0_p)
ax.set_ylim(-2*P_A_value,2*P_A_value)
ax.set_zlim(10*B0,-10*B0)
ax.set_title(r"Flow on the $H=0$ shell ($\kappa_a \neq 0$)" + "\n"
             "red: unstable manifolds of W/S (fwd) · green: stable manifolds (bwd) · "
             "blue/gray: sampled flow lines on H=0 · hollow: M+/M- (off shell)")

# legend proxies
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], color="crimson", lw=2, label="unstable manifold pieces"),
    Line2D([0], [0], color="darkgreen", lw=2, label="stable manifold pieces"),
    Line2D([0], [0], color="steelblue", lw=1, label="flow line (forward)"),
    Line2D([0], [0], color="0.75", lw=1, label="flow line (backward)"),
]
ax.legend(handles=legend_elems, loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()
#plt.savefig("flow_3d.png", dpi=180)
#print("Saved flow_3d.png")